# Vesuvius Challenge 2025 - Enhanced Training V5 (Target: 0.70+)

## Key Improvements Over V4:
| Component | V4 (0.53) | V5 Enhanced |
|-----------|-----------|-------------|
| Patch Size | 128³ | **176³** (optimal context) |
| Loss Function | Dice+BCE+Skeleton | **Dice+BCE+Skeleton+clDice (balanced)** |
| Skeleton Weight | 0.1 | **0.15** (stronger topology) |
| clDice Weight | 0.0 | **0.05** (light, for connectivity) |
| Epochs | 600 | **800** (more convergence) |
| Augmentation | Basic | **nnUNet + Elastic deformation** |
| Post-processing | Simple threshold | **Topology-aware skeleton filter** |

## Architecture:
- 6-stage ResEncUNet3D (111M params)
- SafeInstanceNorm3d (AMP-safe, prevents NaN)
- scSE attention modules
- Deep supervision (multi-scale loss)
- Gradient checkpointing (memory efficient)

## Training Strategy:
1. **Phase 1 (Epochs 0-200)**: Dice + BCE only → learn basic segmentation
2. **Phase 2 (Epochs 200-500)**: Add Skeleton loss (0.15) → learn topology
3. **Phase 3 (Epochs 500-800)**: Add clDice (0.05) → refine connectivity
4. **LR Schedule**: Polynomial decay (0.9 power) with warmup

## Expected Score Breakdown:
- SurfaceDice@τ: 0.75-0.80 (from larger patches + better loss)
- VOI_score: 0.70-0.75 (from skeleton loss)
- TopoScore: 0.65-0.70 (from clDice + topology-aware postproc)
- **Total: 0.70-0.75**

In [ ]:
# Install required packages
!pip install imagecodecs connected-components-3d tifffile scikit-image -q

In [ ]:
# =============================================================================
# CELL 1: IMPORTS & CONFIGURATION - V5 ENHANCED
# =============================================================================

import os
import gc
import json
import math
import random
import warnings
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint

from scipy import ndimage
from scipy.ndimage import distance_transform_edt, gaussian_filter
from skimage.morphology import skeletonize_3d, remove_small_objects

try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False
    print("⚠️ cc3d not found, using scipy (slower)")

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION - V5 ENHANCED (OPTIMIZED FOR 0.70+ SCORE)
# =============================================================================

@dataclass
class Config:
    # Data paths
    DATA_ROOT: Path = Path("/kaggle/input/3d-volume-training-data")
    CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints_v5_enhanced")
    LOAD_CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints_v5_enhanced")
    
    # ==========================================================================
    # MODEL ARCHITECTURE - OPTIMIZED
    # ==========================================================================
    PATCH_SIZE: Tuple[int, int, int] = (176, 176, 176)  # Optimal: larger context
    FEATURES: List[int] = None  # [32, 64, 128, 256, 320, 320]
    N_BLOCKS: List[int] = None  # [1, 3, 4, 6, 6, 6]
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    USE_GRADIENT_CHECKPOINTING: bool = True  # Save memory
    
    # ==========================================================================
    # TRAINING SETTINGS - EXTENDED FOR BETTER CONVERGENCE
    # ==========================================================================
    EPOCHS: int = 800  # More training time
    BATCH_SIZE: int = 1  # Reduced for larger patches (176³)
    ACCUMULATION_STEPS: int = 4  # Effective batch size = 4
    NUM_WORKERS: int = 8
    LEARNING_RATE: float = 0.01
    MOMENTUM: float = 0.99
    WEIGHT_DECAY: float = 3e-5
    GRADIENT_CLIP: float = 5.0
    WARMUP_EPOCHS: int = 20  # LR warmup
    
    # ==========================================================================
    # LOSS WEIGHTS - BALANCED FOR ALL 3 METRICS
    # ==========================================================================
    DICE_WEIGHT: float = 0.5
    BCE_WEIGHT: float = 0.5
    SKELETON_WEIGHT: float = 0.15  # Increased from 0.1 → better topology
    CLDICE_WEIGHT: float = 0.05    # Light clDice → connectivity boost
    
    # ==========================================================================
    # PROGRESSIVE LOSS SCHEDULING - 3 PHASES
    # ==========================================================================
    # Phase 1: Learn basic segmentation
    SKELETON_START_EPOCH: int = 200  # Start after basics learned
    SKELETON_WARMUP_EPOCHS: int = 100  # Gradual introduction
    
    # Phase 2: Learn topology
    CLDICE_START_EPOCH: int = 500  # Start after skeleton learned
    CLDICE_WARMUP_EPOCHS: int = 100
    
    # Training control
    RESUME_TRAINING: bool = True  # Set False for fresh start
    LOAD_BEST: bool = False
    VALIDATE_EVERY: int = 10
    SAVE_EVERY: int = 20
    OVERRIDE_LR: float = 0.0
    
    # Device
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True
    
    # ==========================================================================
    # DATA SETTINGS - OPTIMIZED SAMPLING
    # ==========================================================================
    DATA_FRACTION: float = 1.0
    PATCHES_PER_VOLUME: int = 16  # More patches per volume
    PRELOAD_ALL: bool = True
    FG_OVERSAMPLE_RATIO: float = 0.5  # 50% foreground-centered
    
    # ==========================================================================
    # ENHANCED AUGMENTATION - nnUNet + Elastic
    # ==========================================================================
    # Spatial
    AUG_ROTATION_RANGE: float = 30.0
    AUG_ROTATION_P: float = 0.3
    AUG_SCALE_RANGE: Tuple[float, float] = (0.7, 1.4)
    AUG_SCALE_P: float = 0.3
    AUG_ELASTIC_P: float = 0.3  # NEW: Elastic deformation
    AUG_ELASTIC_ALPHA: float = 50.0
    AUG_ELASTIC_SIGMA: float = 5.0
    
    # Intensity
    AUG_GAUSSIAN_NOISE_P: float = 0.15
    AUG_GAUSSIAN_NOISE_STD: Tuple[float, float] = (0.0, 0.1)
    AUG_GAUSSIAN_BLUR_P: float = 0.25
    AUG_GAUSSIAN_BLUR_SIGMA: Tuple[float, float] = (0.5, 1.5)
    AUG_BRIGHTNESS_P: float = 0.2
    AUG_BRIGHTNESS_RANGE: float = 0.3
    AUG_CONTRAST_P: float = 0.2
    AUG_CONTRAST_RANGE: Tuple[float, float] = (0.7, 1.3)
    AUG_GAMMA_P: float = 0.3
    AUG_GAMMA_RANGE: Tuple[float, float] = (0.7, 1.5)
    AUG_GAMMA_INVERT_P: float = 0.1
    AUG_LOW_RES_P: float = 0.25
    AUG_LOW_RES_ZOOM: Tuple[float, float] = (0.5, 1.0)
    AUG_MIRROR_P: float = 0.5
    
    SEED: int = 42
    
    def __post_init__(self):
        if self.FEATURES is None:
            self.FEATURES = [32, 64, 128, 256, 320, 320]
        if self.N_BLOCKS is None:
            self.N_BLOCKS = [1, 3, 4, 6, 6, 6]
        self.CHECKPOINT_DIR = Path(self.CHECKPOINT_DIR)
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        self.LOAD_CHECKPOINT_DIR = Path(self.LOAD_CHECKPOINT_DIR)

cfg = Config()
cfg.__post_init__()

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False

set_seed(cfg.SEED)

print("="*80)
print(" "*20 + "V5 ENHANCED TRAINING - TARGET: 0.70+")
print("="*80)
print(f"\n🎯 TARGET SCORE BREAKDOWN:")
print(f"   SurfaceDice@τ (35%): 0.75-0.80 → larger patches + better loss")
print(f"   VOI_score (35%):     0.70-0.75 → skeleton loss")
print(f"   TopoScore (30%):     0.65-0.70 → clDice + topology postproc")
print(f"   " + "="*60)
print(f"   EXPECTED TOTAL:      0.70-0.75")
print(f"\n📦 Configuration:")
print(f"   Patch size: {cfg.PATCH_SIZE} (vs 128³ in V4)")
print(f"   Effective batch: {cfg.BATCH_SIZE * cfg.ACCUMULATION_STEPS}")
print(f"   Epochs: {cfg.EPOCHS}")
print(f"   AMP: {cfg.USE_AMP} with SafeInstanceNorm3d")
print(f"   Gradient checkpointing: {cfg.USE_GRADIENT_CHECKPOINTING}")
print(f"\n🎲 Loss Schedule (Progressive):")
print(f"   Phase 1 (0-{cfg.SKELETON_START_EPOCH}): Dice + BCE")
print(f"   Phase 2 ({cfg.SKELETON_START_EPOCH}-{cfg.CLDICE_START_EPOCH}): + Skeleton (0.15)")
print(f"   Phase 3 ({cfg.CLDICE_START_EPOCH}-{cfg.EPOCHS}): + clDice (0.05)")
print(f"\n🔧 Augmentation:")
print(f"   nnUNet pipeline + Elastic deformation")
print(f"   Foreground oversample: {cfg.FG_OVERSAMPLE_RATIO*100:.0f}%")
print("="*80)

In [ ]:
# =============================================================================
# CELL 2: UTILITY FUNCTIONS
# =============================================================================

def load_volume_fast(path: Path) -> np.ndarray:
    """Load 3D TIF volume efficiently."""
    return tifffile.imread(str(path))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class CheckpointManager:
    """Manage checkpoint saving/loading with best model tracking."""
    
    def __init__(self, save_dir: Path, load_dir: Optional[Path] = None):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.load_dir = Path(load_dir) if load_dir else self.save_dir
        self.best_score = -float('inf')
        self.best_epoch = 0
    
    def save(self, model, optimizer, scheduler, scaler, epoch, metrics):
        # Always save latest
        ckpt = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'scaler_state_dict': scaler.state_dict(),
            'metrics': metrics,
            'best_score': self.best_score,
            'best_epoch': self.best_epoch,
        }
        
        torch.save(ckpt, self.save_dir / "latest.pth")
        
        # Save best based on validation dice (or train dice if no val)
        score = metrics.get('val_dice', metrics.get('train_dice_loss', 0))
        if score > self.best_score:
            self.best_score = score
            self.best_epoch = epoch
            torch.save(ckpt, self.save_dir / "best_model.pth")
            print(f"   💾 New best model saved! Score: {score:.4f}")
    
    def load(self, model, optimizer, scheduler, scaler, load_best=False):
        filename = "best_model.pth" if load_best else "latest.pth"
        ckpt_path = self.load_dir / filename
        
        if not ckpt_path.exists():
            print(f"   No checkpoint found at {ckpt_path}")
            return 0
        
        print(f"   Loading checkpoint: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=cfg.DEVICE)
        
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if scheduler and ckpt.get('scheduler_state_dict'):
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        scaler.load_state_dict(ckpt['scaler_state_dict'])
        
        self.best_score = ckpt.get('best_score', -float('inf'))
        self.best_epoch = ckpt.get('best_epoch', 0)
        
        epoch = ckpt['epoch']
        print(f"   ✅ Resumed from epoch {epoch} (best: {self.best_score:.4f} @ {self.best_epoch})")
        return epoch


print("Utilities ready")

In [ ]:
# =============================================================================
# CELL 3: ENHANCED AUGMENTATION PIPELINE
# =============================================================================

def elastic_deformation_3d(image, label, alpha, sigma):
    """Apply elastic deformation to 3D volume."""
    shape = image.shape
    
    # Generate random displacement fields
    dz = gaussian_filter((np.random.rand(*shape) * 2 - 1), sigma) * alpha
    dy = gaussian_filter((np.random.rand(*shape) * 2 - 1), sigma) * alpha
    dx = gaussian_filter((np.random.rand(*shape) * 2 - 1), sigma) * alpha
    
    # Create meshgrid
    z, y, x = np.meshgrid(
        np.arange(shape[0]),
        np.arange(shape[1]),
        np.arange(shape[2]),
        indexing='ij'
    )
    
    # Apply displacement
    indices = (
        np.clip(z + dz, 0, shape[0]-1),
        np.clip(y + dy, 0, shape[1]-1),
        np.clip(x + dx, 0, shape[2]-1)
    )
    
    image_def = ndimage.map_coordinates(image, indices, order=1, mode='reflect')
    label_def = ndimage.map_coordinates(label, indices, order=0, mode='constant', cval=0)
    
    return image_def.astype(np.float32), label_def.astype(np.uint8)


def augment_spatial(image, label, cfg):
    """Spatial augmentation: rotation + scaling."""
    # Rotation
    if random.random() < cfg.AUG_ROTATION_P:
        angle = random.uniform(-cfg.AUG_ROTATION_RANGE, cfg.AUG_ROTATION_RANGE)
        axes = random.choice([(0, 1), (0, 2), (1, 2)])
        image = ndimage.rotate(image, angle, axes=axes, reshape=False, order=1, mode='reflect')
        label = ndimage.rotate(label, angle, axes=axes, reshape=False, order=0, mode='constant', cval=0)
    
    # Scaling
    if random.random() < cfg.AUG_SCALE_P:
        scale = random.uniform(*cfg.AUG_SCALE_RANGE)
        shape = image.shape
        zoom_factors = [scale] * 3
        
        image = ndimage.zoom(image, zoom_factors, order=1, mode='reflect')
        label = ndimage.zoom(label, zoom_factors, order=0, mode='constant', cval=0)
        
        # Crop or pad to original size
        new_shape = image.shape
        result_img = np.zeros(shape, dtype=np.float32)
        result_lbl = np.zeros(shape, dtype=np.uint8)
        
        slices_src = tuple(slice(0, min(s, ns)) for s, ns in zip(shape, new_shape))
        slices_dst = tuple(slice(0, min(s, ns)) for s, ns in zip(shape, new_shape))
        
        result_img[slices_dst] = image[slices_src]
        result_lbl[slices_dst] = label[slices_src]
        
        image, label = result_img, result_lbl
    
    return image, label


def augment_gaussian_noise(image, cfg):
    if random.random() >= cfg.AUG_GAUSSIAN_NOISE_P:
        return image
    std = random.uniform(*cfg.AUG_GAUSSIAN_NOISE_STD)
    noise = np.random.normal(0, std, image.shape).astype(np.float32)
    return image + noise


def augment_gaussian_blur(image, cfg):
    if random.random() >= cfg.AUG_GAUSSIAN_BLUR_P:
        return image
    sigma = random.uniform(*cfg.AUG_GAUSSIAN_BLUR_SIGMA)
    return gaussian_filter(image, sigma).astype(np.float32)


def augment_brightness(image, cfg):
    if random.random() >= cfg.AUG_BRIGHTNESS_P:
        return image
    factor = random.uniform(-cfg.AUG_BRIGHTNESS_RANGE, cfg.AUG_BRIGHTNESS_RANGE)
    return image + factor


def augment_contrast(image, cfg):
    if random.random() >= cfg.AUG_CONTRAST_P:
        return image
    factor = random.uniform(*cfg.AUG_CONTRAST_RANGE)
    mean = image.mean()
    return (image - mean) * factor + mean


def augment_gamma(image, cfg):
    if random.random() >= cfg.AUG_GAMMA_P:
        return image
    
    gamma = random.uniform(*cfg.AUG_GAMMA_RANGE)
    if random.random() < cfg.AUG_GAMMA_INVERT_P:
        gamma = -gamma
    
    mn = image.min()
    rng = image.max() - mn
    
    if rng > 0:
        if gamma < 0:
            image = -image
            gamma = -gamma
        image = rng - ((rng - (image - mn)) / rng) ** gamma * rng + mn
    
    return image


def augment_low_resolution(image, cfg):
    if random.random() >= cfg.AUG_LOW_RES_P:
        return image
    
    axis = random.randint(0, 2)
    zoom_factor = random.uniform(*cfg.AUG_LOW_RES_ZOOM)
    
    if zoom_factor >= 0.99:
        return image
    
    shape = list(image.shape)
    zoom = [1.0, 1.0, 1.0]
    zoom[axis] = zoom_factor
    
    downsampled = ndimage.zoom(image, zoom, order=0)
    zoom_back = [1.0, 1.0, 1.0]
    zoom_back[axis] = shape[axis] / downsampled.shape[axis]
    upsampled = ndimage.zoom(downsampled, zoom_back, order=3)
    
    if upsampled.shape != tuple(shape):
        result = np.zeros(shape, dtype=np.float32)
        slices = tuple(slice(0, min(s, u)) for s, u in zip(shape, upsampled.shape))
        result[slices] = upsampled[slices]
        return result
    
    return upsampled.astype(np.float32)


def augment_mirror(image, label, cfg):
    for axis in range(3):
        if random.random() < cfg.AUG_MIRROR_P:
            image = np.flip(image, axis)
            label = np.flip(label, axis)
    return np.ascontiguousarray(image), np.ascontiguousarray(label)


def apply_enhanced_augmentations(image, label, cfg):
    """Enhanced augmentation pipeline with elastic deformation."""
    # Spatial augmentations
    image, label = augment_spatial(image, label, cfg)
    
    # Elastic deformation (NEW!)
    if random.random() < cfg.AUG_ELASTIC_P:
        image, label = elastic_deformation_3d(
            image, label, 
            cfg.AUG_ELASTIC_ALPHA, 
            cfg.AUG_ELASTIC_SIGMA
        )
    
    # Intensity augmentations
    image = augment_gaussian_noise(image, cfg)
    image = augment_gaussian_blur(image, cfg)
    image = augment_brightness(image, cfg)
    image = augment_contrast(image, cfg)
    image = augment_gamma(image, cfg)
    image = augment_low_resolution(image, cfg)
    
    # Mirror
    image, label = augment_mirror(image, label, cfg)
    
    return image.astype(np.float32), label


print("Enhanced augmentation pipeline ready (nnUNet + Elastic)")

In [ ]:
# =============================================================================
# CELL 4: DATASET WITH OPTIMIZED SAMPLING
# =============================================================================

class VesuviusDatasetV5(Dataset):
    """Enhanced dataset with better foreground sampling."""
    
    def __init__(
        self,
        csv_path: Path,
        images_dir: Path,
        labels_dir: Path,
        patch_size: Tuple[int, int, int] = (176, 176, 176),
        is_train: bool = True,
        data_fraction: float = 1.0,
        patches_per_volume: int = 16,
        preload: bool = True,
    ):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        
        # Load and filter valid volumes
        df = pd.read_csv(csv_path)
        valid_ids = []
        for idx in df['id'].values:
            if (self.images_dir / f"{idx}.tif").exists() and \
               (self.labels_dir / f"{idx}.tif").exists():
                valid_ids.append(idx)
        
        if data_fraction < 1.0:
            n = max(1, int(len(valid_ids) * data_fraction))
            random.shuffle(valid_ids)
            valid_ids = valid_ids[:n]
        
        self.volume_ids = valid_ids
        self.volumes = {}
        self.fg_coords = {}
        
        # Preload all data to RAM
        if preload:
            print(f"Preloading {len(self.volume_ids)} volumes to RAM...")
            for vol_id in tqdm(self.volume_ids, desc="Loading"):
                img = load_volume_fast(self.images_dir / f"{vol_id}.tif").astype(np.float32)
                lbl = load_volume_fast(self.labels_dir / f"{vol_id}.tif").astype(np.uint8)
                
                # Normalize
                img = (img - img.mean()) / (img.std() + 1e-8)
                self.volumes[vol_id] = (img, lbl)
                
                # Cache foreground coordinates for smart sampling
                fg = np.argwhere(lbl == 1)
                if len(fg) > 10000:
                    fg = fg[np.random.choice(len(fg), 10000, replace=False)]
                self.fg_coords[vol_id] = fg if len(fg) > 0 else None
            
            # Memory usage estimate
            sample_vol = next(iter(self.volumes.values()))
            vol_size_mb = (sample_vol[0].nbytes + sample_vol[1].nbytes) / 1e6
            total_gb = vol_size_mb * len(self.volumes) / 1000
            print(f"Loaded {len(self.volumes)} volumes ({total_gb:.1f} GB in RAM)")
        
        print(f"Dataset ready: {len(self)} samples")
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def __getitem__(self, idx):
        vol_idx = idx // self.patches_per_volume
        vol_id = self.volume_ids[vol_idx]
        
        image, label = self.volumes[vol_id]
        d, h, w = image.shape
        pd, ph, pw = self.patch_size
        
        # Pad if needed
        if d < pd or h < ph or w < pw:
            pad_d = max(0, pd - d)
            pad_h = max(0, ph - h)
            pad_w = max(0, pw - w)
            image = np.pad(image, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='reflect')
            label = np.pad(label, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='constant', constant_values=2)
            d, h, w = image.shape
        
        # Smart sampling: foreground-centered 50% of the time
        fg = self.fg_coords.get(vol_id)
        force_fg = self.is_train and random.random() < cfg.FG_OVERSAMPLE_RATIO and fg is not None and len(fg) > 0
        
        if force_fg:
            c = fg[random.randint(0, len(fg) - 1)]
            z = max(0, min(c[0] - pd // 2, d - pd))
            y = max(0, min(c[1] - ph // 2, h - ph))
            x = max(0, min(c[2] - pw // 2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))
        
        img_patch = image[z:z+pd, y:y+ph, x:x+pw].copy()
        lbl_patch = label[z:z+pd, y:y+ph, x:x+pw].copy()
        
        # Apply augmentation
        if self.is_train:
            img_patch, lbl_patch = apply_enhanced_augmentations(img_patch, lbl_patch, cfg)
        
        fg_mask = (lbl_patch == 1).astype(np.float32)
        ig_mask = (lbl_patch == 2).astype(np.float32)
        
        return {
            'image': torch.from_numpy(img_patch).unsqueeze(0).float(),
            'label': torch.from_numpy(fg_mask).unsqueeze(0).float(),
            'ignore_mask': torch.from_numpy(ig_mask).unsqueeze(0).float(),
        }


print("Enhanced dataset ready (176³ patches, smart FG sampling)")

In [ ]:
# =============================================================================
# CELL 5: MODEL ARCHITECTURE (SafeInstanceNorm3d + Gradient Checkpointing)
# =============================================================================

class SafeInstanceNorm3d(nn.Module):
    """AMP-safe InstanceNorm3d that prevents NaN from small variances."""
    
    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        
        if self.affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
        else:
            self.register_parameter('weight', None)
            self.register_parameter('bias', None)
    
    def forward(self, x):
        mean = x.mean(dim=[2, 3, 4], keepdim=True)
        var = x.var(dim=[2, 3, 4], keepdim=True, unbiased=False)
        var_safe = torch.clamp(var, min=self.eps)  # Prevent division by zero
        x_norm = (x - mean) / torch.sqrt(var_safe + self.eps)
        
        if self.affine:
            x_norm = self.weight.view(1, -1, 1, 1, 1) * x_norm + self.bias.view(1, -1, 1, 1, 1)
        
        return x_norm


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=True),
            SafeInstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[ConvBlock(channels, channels) for _ in range(n_convs)]
        )
    
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    """Spatial and Channel Squeeze & Excitation."""
    
    def __init__(self, channels, reduction=16):
        super().__init__()
        # Channel SE
        self.cSE = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Conv3d(channels, channels // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv3d(channels // reduction, channels, 1),
            nn.Sigmoid()
        )
        # Spatial SE
        self.sSE = nn.Sequential(
            nn.Conv3d(channels, 1, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return x * self.cSE(x) + x * self.sSE(x)


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_blocks, use_scse=True):
        super().__init__()
        self.down = ConvBlock(in_ch, out_ch)
        self.res_blocks = nn.Sequential(
            *[ResBlock(out_ch, n_convs=2) for _ in range(n_blocks)]
        )
        self.scse = scSEBlock(out_ch) if use_scse else nn.Identity()
    
    def forward(self, x):
        x = self.down(x)
        x = self.res_blocks(x)
        x = self.scse(x)
        return x


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, n_blocks, use_scse=True):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, in_ch, 2, stride=2)
        self.conv = ConvBlock(in_ch + skip_ch, out_ch)
        self.res_blocks = nn.Sequential(
            *[ResBlock(out_ch, n_convs=2) for _ in range(n_blocks)]
        )
        self.scse = scSEBlock(out_ch) if use_scse else nn.Identity()
    
    def forward(self, x, skip):
        x = self.up(x)
        # Handle size mismatch
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        x = self.res_blocks(x)
        x = self.scse(x)
        return x


class ResEncUNet3D(nn.Module):
    """6-stage ResEncUNet3D with optional gradient checkpointing."""
    
    def __init__(
        self,
        features=[32, 64, 128, 256, 320, 320],
        n_blocks=[1, 3, 4, 6, 6, 6],
        use_scse=True,
        use_deep_supervision=True,
        use_gradient_checkpointing=False,
    ):
        super().__init__()
        self.use_deep_supervision = use_deep_supervision
        self.use_gradient_checkpointing = use_gradient_checkpointing
        
        # Encoder
        self.enc0 = EncoderBlock(1, features[0], n_blocks[0], use_scse)
        self.pool0 = nn.MaxPool3d(2)
        
        self.enc1 = EncoderBlock(features[0], features[1], n_blocks[1], use_scse)
        self.pool1 = nn.MaxPool3d(2)
        
        self.enc2 = EncoderBlock(features[1], features[2], n_blocks[2], use_scse)
        self.pool2 = nn.MaxPool3d(2)
        
        self.enc3 = EncoderBlock(features[2], features[3], n_blocks[3], use_scse)
        self.pool3 = nn.MaxPool3d(2)
        
        self.enc4 = EncoderBlock(features[3], features[4], n_blocks[4], use_scse)
        self.pool4 = nn.MaxPool3d(2)
        
        # Bottleneck
        self.bottleneck = EncoderBlock(features[4], features[5], n_blocks[5], use_scse)
        
        # Decoder
        self.dec4 = DecoderBlock(features[5], features[4], features[4], n_blocks[4], use_scse)
        self.dec3 = DecoderBlock(features[4], features[3], features[3], n_blocks[3], use_scse)
        self.dec2 = DecoderBlock(features[3], features[2], features[2], n_blocks[2], use_scse)
        self.dec1 = DecoderBlock(features[2], features[1], features[1], n_blocks[1], use_scse)
        self.dec0 = DecoderBlock(features[1], features[0], features[0], n_blocks[0], use_scse)
        
        # Output heads
        self.out_main = nn.Conv3d(features[0], 1, 1)
        
        if use_deep_supervision:
            self.out_deep4 = nn.Conv3d(features[4], 1, 1)
            self.out_deep3 = nn.Conv3d(features[3], 1, 1)
            self.out_deep2 = nn.Conv3d(features[2], 1, 1)
    
    def _apply_checkpoint(self, module, x):
        """Apply gradient checkpointing if enabled."""
        if self.use_gradient_checkpointing and self.training:
            return checkpoint(module, x, use_reentrant=False)
        return module(x)
    
    def forward(self, x):
        # Encoder with gradient checkpointing
        e0 = self._apply_checkpoint(self.enc0, x)
        p0 = self.pool0(e0)
        
        e1 = self._apply_checkpoint(self.enc1, p0)
        p1 = self.pool1(e1)
        
        e2 = self._apply_checkpoint(self.enc2, p1)
        p2 = self.pool2(e2)
        
        e3 = self._apply_checkpoint(self.enc3, p2)
        p3 = self.pool3(e3)
        
        e4 = self._apply_checkpoint(self.enc4, p3)
        p4 = self.pool4(e4)
        
        # Bottleneck
        b = self._apply_checkpoint(self.bottleneck, p4)
        
        # Decoder
        d4 = self._apply_checkpoint(lambda x: self.dec4(x, e4), b)
        d3 = self._apply_checkpoint(lambda x: self.dec3(x, e3), d4)
        d2 = self._apply_checkpoint(lambda x: self.dec2(x, e2), d3)
        d1 = self._apply_checkpoint(lambda x: self.dec1(x, e1), d2)
        d0 = self._apply_checkpoint(lambda x: self.dec0(x, e0), d1)
        
        # Main output
        out = self.out_main(d0)
        
        # Deep supervision (only during training)
        if self.use_deep_supervision and self.training:
            out_d4 = self.out_deep4(d4)
            out_d3 = self.out_deep3(d3)
            out_d2 = self.out_deep2(d2)
            return out, out_d4, out_d3, out_d2
        
        return out


print("Model architecture ready (SafeInstanceNorm3d + Gradient Checkpointing)")

In [ ]:
# =============================================================================
# CELL 6: LOSS FUNCTIONS (Balanced for all 3 metrics)
# =============================================================================

class DiceLoss(nn.Module):
    """Dice loss with ignore mask support."""
    
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, ignore_mask=None):
        pred = torch.sigmoid(pred)
        
        if ignore_mask is not None:
            pred = pred * (1 - ignore_mask)
            target = target * (1 - ignore_mask)
        
        pred_flat = pred.view(-1)
        target_flat = target.view(-1)
        
        intersection = (pred_flat * target_flat).sum()
        union = pred_flat.sum() + target_flat.sum()
        
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice


class BCELoss(nn.Module):
    """Binary cross-entropy with ignore mask support."""
    
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
    
    def forward(self, pred, target, ignore_mask=None):
        loss = self.bce(pred, target)
        
        if ignore_mask is not None:
            loss = loss * (1 - ignore_mask)
        
        return loss.mean()


class SkeletonRecallLoss(nn.Module):
    """Skeleton Recall loss for topology preservation."""
    
    def __init__(self, min_skeleton_pixels=10):
        super().__init__()
        self.min_skeleton_pixels = min_skeleton_pixels
    
    def forward(self, pred, target, ignore_mask=None):
        pred = torch.sigmoid(pred)
        
        if ignore_mask is not None:
            pred = pred * (1 - ignore_mask)
            target = target * (1 - ignore_mask)
        
        # Compute skeleton on CPU (skeletonize_3d not differentiable)
        batch_loss = 0.0
        batch_size = target.shape[0]
        
        for b in range(batch_size):
            tgt_np = target[b, 0].detach().cpu().numpy()
            
            # Skip if no foreground
            if tgt_np.sum() < self.min_skeleton_pixels:
                continue
            
            # Compute skeleton
            tgt_binary = (tgt_np > 0.5).astype(np.uint8)
            
            try:
                skeleton = skeletonize_3d(tgt_binary)
                skeleton_coords = np.argwhere(skeleton)
                
                if len(skeleton_coords) == 0:
                    continue
                
                # Sample skeleton points (max 1000 for speed)
                if len(skeleton_coords) > 1000:
                    indices = np.random.choice(len(skeleton_coords), 1000, replace=False)
                    skeleton_coords = skeleton_coords[indices]
                
                # Get prediction values at skeleton points
                pred_vals = pred[b, 0, skeleton_coords[:, 0], skeleton_coords[:, 1], skeleton_coords[:, 2]]
                
                # Skeleton recall: average prediction on skeleton
                skeleton_recall = pred_vals.mean()
                batch_loss += (1.0 - skeleton_recall)
                
            except:
                # Skip on error
                continue
        
        return batch_loss / max(batch_size, 1)


class SoftClDiceLoss(nn.Module):
    """Soft centerline Dice loss for connectivity."""
    
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def soft_skeletonize(self, x, iterations=10):
        """Soft skeletonization using iterative thinning."""
        for _ in range(iterations):
            # Simple soft thinning operation
            x = F.max_pool3d(x, kernel_size=3, stride=1, padding=1)
        return x
    
    def forward(self, pred, target, ignore_mask=None):
        pred = torch.sigmoid(pred)
        
        if ignore_mask is not None:
            pred = pred * (1 - ignore_mask)
            target = target * (1 - ignore_mask)
        
        # Soft skeletonization
        pred_skel = self.soft_skeletonize(pred, iterations=7)
        target_skel = self.soft_skeletonize(target, iterations=7)
        
        # Tprec = intersection(pred_skel, target) / pred_skel
        tprec_num = (pred_skel * target).sum()
        tprec_den = pred_skel.sum()
        tprec = (tprec_num + self.smooth) / (tprec_den + self.smooth)
        
        # Tsens = intersection(target_skel, pred) / target_skel
        tsens_num = (target_skel * pred).sum()
        tsens_den = target_skel.sum()
        tsens = (tsens_num + self.smooth) / (tsens_den + self.smooth)
        
        # clDice = 2 * Tprec * Tsens / (Tprec + Tsens)
        cldice = (2.0 * tprec * tsens) / (tprec + tsens + self.smooth)
        
        return 1.0 - cldice


class CombinedLoss(nn.Module):
    """Combined loss with progressive scheduling."""
    
    def __init__(
        self,
        dice_weight=0.5,
        bce_weight=0.5,
        skeleton_weight=0.15,
        cldice_weight=0.05,
        skeleton_start=200,
        skeleton_warmup=100,
        cldice_start=500,
        cldice_warmup=100,
    ):
        super().__init__()
        self.dice_loss = DiceLoss()
        self.bce_loss = BCELoss()
        self.skeleton_loss = SkeletonRecallLoss()
        self.cldice_loss = SoftClDiceLoss()
        
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.skeleton_weight = skeleton_weight
        self.cldice_weight = cldice_weight
        
        self.skeleton_start = skeleton_start
        self.skeleton_warmup = skeleton_warmup
        self.cldice_start = cldice_start
        self.cldice_warmup = cldice_warmup
        
        self.current_epoch = 0
    
    def get_schedule_weight(self, epoch, start_epoch, warmup_epochs, max_weight):
        """Progressive weight scheduling."""
        if epoch < start_epoch:
            return 0.0
        elif epoch < start_epoch + warmup_epochs:
            progress = (epoch - start_epoch) / warmup_epochs
            return max_weight * progress
        else:
            return max_weight
    
    def forward(self, pred, target, ignore_mask=None, epoch=0):
        self.current_epoch = epoch
        
        # Base losses (always active)
        dice = self.dice_loss(pred, target, ignore_mask)
        bce = self.bce_loss(pred, target, ignore_mask)
        
        total_loss = self.dice_weight * dice + self.bce_weight * bce
        
        loss_dict = {
            'dice': dice.item(),
            'bce': bce.item(),
        }
        
        # Skeleton loss (progressive)
        skel_weight = self.get_schedule_weight(
            epoch, self.skeleton_start, self.skeleton_warmup, self.skeleton_weight
        )
        if skel_weight > 0:
            skeleton = self.skeleton_loss(pred, target, ignore_mask)
            total_loss += skel_weight * skeleton
            loss_dict['skeleton'] = skeleton.item()
            loss_dict['skel_w'] = skel_weight
        
        # clDice loss (progressive)
        cldice_w = self.get_schedule_weight(
            epoch, self.cldice_start, self.cldice_warmup, self.cldice_weight
        )
        if cldice_w > 0:
            cldice = self.cldice_loss(pred, target, ignore_mask)
            total_loss += cldice_w * cldice
            loss_dict['cldice'] = cldice.item()
            loss_dict['cldice_w'] = cldice_w
        
        loss_dict['total'] = total_loss.item()
        
        return total_loss, loss_dict


print("Loss functions ready (Dice + BCE + Skeleton + clDice with progressive scheduling)")

In [ ]:
# =============================================================================
# CELL 7: TRAINING & VALIDATION FUNCTIONS
# =============================================================================

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    model.train()
    
    total_loss = 0
    loss_components = {'dice': 0, 'bce': 0, 'skeleton': 0, 'cldice': 0}
    n_batches = 0
    
    optimizer.zero_grad()
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}", leave=False)
    
    for batch_idx, batch in enumerate(pbar):
        image = batch['image'].to(device)
        label = batch['label'].to(device)
        ignore_mask = batch['ignore_mask'].to(device)
        
        # Forward pass with AMP
        with autocast(enabled=cfg.USE_AMP):
            outputs = model(image)
            
            # Handle deep supervision
            if isinstance(outputs, tuple):
                pred_main = outputs[0]
                preds_deep = outputs[1:]
                
                # Main loss
                loss, loss_dict = criterion(pred_main, label, ignore_mask, epoch)
                
                # Deep supervision losses (downsampled targets)
                for i, pred_deep in enumerate(preds_deep):
                    scale = 2 ** (i + 1)
                    label_down = F.avg_pool3d(label, scale)
                    ignore_down = F.max_pool3d(ignore_mask, scale)
                    
                    deep_loss, _ = criterion(pred_deep, label_down, ignore_down, epoch)
                    loss += 0.5 * deep_loss  # Weight deep supervision less
            else:
                loss, loss_dict = criterion(outputs, label, ignore_mask, epoch)
        
        # Backward pass with gradient accumulation
        scaler.scale(loss).backward()
        
        if (batch_idx + 1) % cfg.ACCUMULATION_STEPS == 0:
            # Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        # Track metrics
        total_loss += loss_dict['total']
        for k in loss_components:
            if k in loss_dict:
                loss_components[k] += loss_dict[k]
        n_batches += 1
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f"{loss_dict['total']:.4f}",
            'dice': f"{loss_dict.get('dice', 0):.4f}",
        })
    
    # Average metrics
    metrics = {'train_loss': total_loss / n_batches}
    for k, v in loss_components.items():
        if v > 0:
            metrics[f'train_{k}'] = v / n_batches
    
    return metrics


def validate(model, loader, device):
    model.eval()
    
    total_dice = 0
    n_samples = 0
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating", leave=False):
            image = batch['image'].to(device)
            label = batch['label'].to(device)
            ignore_mask = batch['ignore_mask'].to(device)
            
            with autocast(enabled=cfg.USE_AMP):
                outputs = model(image)
                if isinstance(outputs, tuple):
                    outputs = outputs[0]
                
                pred = torch.sigmoid(outputs)
            
            # Compute dice per sample
            for b in range(pred.shape[0]):
                p = pred[b, 0]
                t = label[b, 0]
                ign = ignore_mask[b, 0]
                
                # Apply ignore mask and threshold
                p_binary = (p > 0.5).float()
                p_binary = p_binary * (1 - ign)
                t = t * (1 - ign)
                
                inter = (p_binary * t).sum()
                union = p_binary.sum() + t.sum()
                
                dice = (2 * inter + 1e-5) / (union + 1e-5)
                total_dice += dice.item()
                n_samples += 1
    
    return {'val_dice': total_dice / max(n_samples, 1)}


print("Training functions ready")

In [ ]:
# =============================================================================
# CELL 8: MAIN TRAINING LOOP
# =============================================================================

def main_training():
    print("\n" + "="*80)
    print(" "*25 + "STARTING V5 ENHANCED TRAINING")
    print("="*80)
    
    # Paths
    train_csv = cfg.DATA_ROOT / "train.csv"
    train_images = cfg.DATA_ROOT / "train_images"
    train_labels = cfg.DATA_ROOT / "train_labels"
    
    # Create datasets
    print("\n[1/5] Loading training data...")
    train_dataset = VesuviusDatasetV5(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=True,
        data_fraction=cfg.DATA_FRACTION,
        patches_per_volume=cfg.PATCHES_PER_VOLUME,
        preload=cfg.PRELOAD_ALL,
    )
    
    print("\n[2/5] Loading validation data...")
    val_dataset = VesuviusDatasetV5(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=False,
        data_fraction=0.15,
        patches_per_volume=2,
        preload=cfg.PRELOAD_ALL,
    )
    
    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )
    
    print(f"\nTrain: {len(train_dataset)} samples, {len(train_loader)} batches/epoch")
    print(f"Val: {len(val_dataset)} samples")
    print(f"Effective batch size: {cfg.BATCH_SIZE * cfg.ACCUMULATION_STEPS}")
    
    # Model
    print("\n[3/5] Creating model...")
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
        use_gradient_checkpointing=cfg.USE_GRADIENT_CHECKPOINTING,
    ).to(cfg.DEVICE)
    
    print(f"Parameters: {count_parameters(model):,}")
    
    # Loss
    criterion = CombinedLoss(
        dice_weight=cfg.DICE_WEIGHT,
        bce_weight=cfg.BCE_WEIGHT,
        skeleton_weight=cfg.SKELETON_WEIGHT,
        cldice_weight=cfg.CLDICE_WEIGHT,
        skeleton_start=cfg.SKELETON_START_EPOCH,
        skeleton_warmup=cfg.SKELETON_WARMUP_EPOCHS,
        cldice_start=cfg.CLDICE_START_EPOCH,
        cldice_warmup=cfg.CLDICE_WARMUP_EPOCHS,
    )
    
    # Optimizer
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=cfg.LEARNING_RATE,
        momentum=cfg.MOMENTUM,
        weight_decay=cfg.WEIGHT_DECAY,
        nesterov=True,
    )
    
    # LR Scheduler with warmup
    def lr_lambda(epoch):
        if epoch < cfg.WARMUP_EPOCHS:
            return (epoch + 1) / cfg.WARMUP_EPOCHS
        else:
            progress = (epoch - cfg.WARMUP_EPOCHS) / (cfg.EPOCHS - cfg.WARMUP_EPOCHS)
            return (1 - progress) ** 0.9
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    scaler = GradScaler(enabled=cfg.USE_AMP)
    
    # Checkpoint manager
    ckpt_mgr = CheckpointManager(
        save_dir=cfg.CHECKPOINT_DIR,
        load_dir=cfg.LOAD_CHECKPOINT_DIR
    )
    
    # Resume if needed
    print("\n[4/5] Checking for checkpoint...")
    if cfg.RESUME_TRAINING:
        start_epoch = ckpt_mgr.load(model, optimizer, scheduler, scaler, cfg.LOAD_BEST)
    else:
        start_epoch = 0
    
    # Override LR if requested
    if cfg.OVERRIDE_LR > 0:
        for param_group in optimizer.param_groups:
            param_group['lr'] = cfg.OVERRIDE_LR
        print(f"   LR overridden to: {cfg.OVERRIDE_LR}")
    
    print(f"\n[5/5] Starting training from epoch {start_epoch + 1}")
    print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")
    print("="*80)
    
    # Training loop
    for epoch in range(start_epoch, cfg.EPOCHS):
        epoch_start = time.time()
        
        # Train
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, cfg.DEVICE, epoch
        )
        
        # Step scheduler
        scheduler.step()
        
        # Validate
        if (epoch + 1) % cfg.VALIDATE_EVERY == 0:
            val_metrics = validate(model, val_loader, cfg.DEVICE)
        else:
            val_metrics = {'val_dice': 0}
        
        epoch_time = time.time() - epoch_start
        lr = optimizer.param_groups[0]['lr']
        
        # Print summary
        print(f"\nEpoch {epoch+1}/{cfg.EPOCHS} | {epoch_time:.1f}s | LR: {lr:.6f}")
        print(f"  Loss: {train_metrics['train_loss']:.4f} | ", end="")
        print(f"Dice: {train_metrics.get('train_dice', 0):.4f} | ", end="")
        if 'train_skeleton' in train_metrics:
            print(f"Skel: {train_metrics['train_skeleton']:.4f} | ", end="")
        if 'train_cldice' in train_metrics:
            print(f"clDice: {train_metrics['train_cldice']:.4f} | ", end="")
        if val_metrics['val_dice'] > 0:
            print(f"Val: {val_metrics['val_dice']:.4f}")
        else:
            print()
        
        # Save checkpoint
        if (epoch + 1) % cfg.SAVE_EVERY == 0:
            ckpt_mgr.save(model, optimizer, scheduler, scaler, epoch, 
                         {**train_metrics, **val_metrics})
    
    print("\n" + "="*80)
    print(f"TRAINING COMPLETE! Best: {ckpt_mgr.best_score:.4f} @ epoch {ckpt_mgr.best_epoch}")
    print("="*80)
    
    return model, ckpt_mgr


print("\n🚀 Ready to train! Run main_training() to start.")
print("\nExpected improvements:")
print("  ✓ 176³ patches → better context → higher SurfaceDice")
print("  ✓ Balanced topology losses → better TopoScore & VOI")
print("  ✓ Progressive scheduling → stable convergence")
print("  ✓ Enhanced augmentation → better generalization")
print("\n🎯 Target: 0.70-0.75 (vs 0.53 in V4)")

In [ ]:
# =============================================================================
# RUN TRAINING
# =============================================================================

if __name__ == "__main__":
    model, ckpt_mgr = main_training()